Question 1:

Task: Implement a FAISS-based Retrieval System that fetches the most relevant document for query-based retrieval.

In [6]:
import faiss
import numpy as np
import google.generativeai as genai

genai.configure(api_key="")

d = 3072
index = faiss.IndexFlatL2(d)

def get_embedding(text):
    # model = genai.GenerativeModel("gemini-3.1-flash-lite-preview")
    # response = model.generate_content(text)
    response = genai.embed_content(model ="models/gemini-embedding-2",content=[text], task_type="retrieval_document")
    return np.array(response['embedding'], dtype='float32').reshape(1, -1)

documents = ["AI is transforming industries", "FAISS is great for vector search", "Generative models are powerful","Agentic AI refers to intelligent systems that autonomously make decisions, plan actions, and execute workflows to achieve specific goals with minimal human supervision."]
doc_embeddings = np.vstack([get_embedding(doc) for doc in documents]).astype('float32')
embedding_dim = doc_embeddings.shape[1] 
print(embedding_dim)
index.add(doc_embeddings)


query = "What is AI Agents?"
query_vector = get_embedding(query)
_, indices = index.search(query_vector, k=1)

retrieved_doc = documents[indices[0][0]]
print("Retrieved Document:", retrieved_doc)


3072
Retrieved Document: Agentic AI refers to intelligent systems that autonomously make decisions, plan actions, and execute workflows to achieve specific goals with minimal human supervision.


Question 2:
Task: Use Gemini API to generate answers based on retrieved knowledge.

In [10]:
import google.generativeai as genai

genai.configure(api_key="")
context = """
RAG (Retrieval-Augmented Generation) is an AI framework that grounds Large Language Models (LLMs) in external, proprietary knowledge bases. 
Instead of relying solely on static training data, 
it searches for relevant information before generating a response, drastically reducing hallucinations.
"""
query = "Define LLM"
model = genai.GenerativeModel(model_name="gemini-3.1-flash-lite-preview")
response = model.generate_content(f"Use the following context to answer: {context}\n\n{query}")
print(response.text)

Based on the context provided, an **LLM** (Large Language Model) is an AI model that relies on static training data to generate responses, which can sometimes lead to hallucinations. RAG is used to ground these models in external, proprietary knowledge bases to improve their accuracy.


Question 3:
Task: Implement FAISS with a hybrid (dense & sparse) vector search for improving information retrieval.

In [19]:
import faiss
import numpy as np
import google.generativeai as genai
from sentence_transformers import SentenceTransformer

genai.configure(api_key="")

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

documents = ["AI is transforming industries",
             "FAISS is great for vector search", 
             "Generative models are powerful",
             "Agentic AI refers to intelligent systems that autonomously make decisions, plan actions, and execute workflows to achieve specific goals with minimal human supervision."]
embeddings = np.vstack([model.encode(doc) for doc in documents], dtype=np.float32)
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)


query = "What is AI Agents?"
query_vector =  np.vstack([model.encode(query)], dtype=np.float32)
D, I = index.search(query_vector, k=1)
retrieved_doc = documents[I[0][0]]
print("Retrieved Document:", retrieved_doc)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

C:\Users\SAIRAM\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SAIRAM\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retrieved Document: Agentic AI refers to intelligent systems that autonomously make decisions, plan actions, and execute workflows to achieve specific goals with minimal human supervision.



Question 4:
Task: Implement a Multi-Turn Conversational Chatbot using Gemini API that maintains context across multiple user interactions.

In [23]:
import google.generativeai as genai
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-3.1-flash-lite-preview")
context = [{"role": "system", 
            "content": "You are a AI Assisstant. Maintain conversational history"}]

def chatbot(user_message):
    context.append({"role": "user", "content": user_message})
    response = model.generate_content(f"{context}\nUser: {user_message}")
    return response.text
print("Chatterbox: Hi I am Sabarmathi. How can i assist you?")
while True:
    user_message = input("You: ")
    if user_message.lower() in ["exit", "quit"]:
        print("Chatterbox: Goodbye!")
        break
    chatting = chatbot(user_message)
    print("Chatterbox:", chatting)
    

Chatterbox: Hi I am Sabarmathi. How can i assist you?


You:  what is AI


Chatterbox: AI, or **Artificial Intelligence**, is a field of computer science that focuses on creating systems capable of performing tasks that typically require human intelligence.

At its core, AI is about building machines or software that can "think" and "learn" to solve problems. Instead of just following a strict set of programmed instructions, AI systems can process large amounts of data, recognize patterns, and make decisions or predictions based on that information.

Here are a few key ways to understand AI:

*   **Machine Learning (ML):** This is the most common form of AI today. It involves training a computer on data so it can improve its performance over time without being explicitly programmed for every scenario. (e.g., teaching an AI to recognize photos of cats by showing it thousands of images of cats).
*   **Generative AI:** This is what you are interacting with right now. It uses models trained on massive datasets to create new content—such as text, images, music, or

You:  uses of AI


Chatterbox: Artificial Intelligence (AI) is the branch of computer science dedicated to creating systems capable of performing tasks that typically require human intelligence. This includes things like reasoning, learning from data, recognizing patterns, understanding language, and solving complex problems.

Since AI can process vast amounts of data much faster than a human, its uses have expanded into nearly every industry today. Here are some of the primary uses of AI:

### 1. Healthcare
*   **Diagnostics:** AI algorithms analyze medical images (like X-rays and MRIs) to detect diseases like cancer earlier and more accurately than human eyes.
*   **Drug Discovery:** Researchers use AI to simulate how molecules interact, drastically shortening the time required to develop new medicines.

### 2. Finance
*   **Fraud Detection:** Banks use AI to monitor transactions in real-time, instantly flagging unusual patterns that might indicate credit card theft or money laundering.
*   **Algorithm

You:  application of it


Chatterbox: Since we’ve discussed what AI is and its general uses, "applications" refers to the specific, real-world fields where AI is currently solving complex problems.

Here are the key sectors where AI is being applied today:

### 1. Healthcare
*   **Diagnostics:** AI algorithms analyze medical imagery (X-rays, MRIs, CT scans) to detect diseases like cancer or fractures often faster and more accurately than humans.
*   **Drug Discovery:** AI models simulate chemical reactions to predict which molecular combinations will be effective for new medications, shortening development cycles from years to months.
*   **Personalized Medicine:** Tailoring treatment plans based on a patient’s specific genetic profile and medical history.

### 2. Finance
*   **Fraud Detection:** AI monitors millions of transactions in real-time to identify patterns indicative of fraud, blocking suspicious activity instantly.
*   **Algorithmic Trading:** Using predictive models to execute high-speed stock trade

You:  exit


Chatterbox: Goodbye!


In [27]:
import faiss
import numpy as np
import google.generativeai as genai
from sentence_transformers import SentenceTransformer

genai.configure(api_key="")

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

documents = ["AI is transforming industries",
             "FAISS is great for vector search", 
             "Generative models are powerful",
             "Agentic AI refers to intelligent systems that autonomously make decisions, plan actions, and execute workflows to achieve specific goals with minimal human supervision."]
embeddings = np.vstack([model.encode(doc) for doc in documents], dtype=np.float32)
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

context = [{"role": "system", 
            "content": "You are a AI Assisstant. Maintain conversational history"}]

def retrieve_doc(user_message):
    query_vector =  np.vstack([model.encode(user_message)], dtype=np.float32)
    D, I = index.search(query_vector, k=1)
    retrieved_doc = documents[I[0][0]]
    return retrieved_doc
    
def chatbot(user_message):
    retrieved_doc = retrieve_doc(user_message)
    full_prompt = f"Use the following retrieved documents to answer the query:\n{retrieved_doc}\n\nQuery: {user_message}"
    chatmodel = genai.GenerativeModel("gemini-3.1-flash-lite-preview")
    response = chatmodel.generate_content(full_prompt)
    return response.text
    
print("Chatterbox: Hi I am Sabarmathi. How can i assist you?")
while True:
    user_message = input("You: ")
    if user_message.lower() in ["exit", "quit"]:
        print("Chatterbox: Goodbye!")
        break
    chatting = chatbot(user_message)
    print("Chatterbox:", chatting)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Chatterbox: Hi I am Sabarmathi. How can i assist you?


You:  what is AI


Chatterbox: Based on the provided information, AI is a transformative technology that is actively reshaping various industries.


You:  what is Agentic AI


Chatterbox: Agentic AI refers to intelligent systems that can autonomously make decisions, plan actions, and execute workflows to achieve specific goals with minimal human supervision.


You:  what is RAG


Chatterbox: Based on the provided document, there is no information available to answer the query "what is RAG." The document only states that "FAISS is great for vector search."


You:  exit


Chatterbox: Goodbye!
